# Kestrel feedback: findings and cleaning decisions

The source investigation found **52 files with incompatible layouts, overlapping versions and different rating scales**. The table below summarises the decisions the ingestion pipeline needs to implement; evidence follows.

| File / source | Finding | Proposed decision |
|---|---|---|
| Geelong | The rating direction reverses between 2025 and 2026. | Apply sheet-specific scale rules; retain the original score. |
| Ringwood Q1 | Row 37 introduces a new template and a ten-point score. | Map the two header segments separately. |
| Chermside Q1 FINAL / v2 | v2 contains the same 112 responses plus six additions. | Count shared references once and preserve both source versions. |
| Penrith | June and July tabs contain identical grids. | Investigate the copied tab; do not invent June dates. |
| Tamworth | Ten feedback rows have no rating; eight still have comments. | Retain the responses; omit absent ratings from rating KPIs. |
| Toowoomba Q4 summary | 71 responses are represented as aggregate counts. | Store aggregates separately from individual feedback. |
| Southport copied reviews | Eight reviews have relative dates. | Keep the source distinct; do not invent exact event dates. |
| Dandenong / Frankston | Title rows, repeated headers and layout changes affect parsing. | Map headers by source segment and classify notes/totals separately. |

**Privacy policy:** raw sources remain local and unchanged. General previews withhold free text and identifiers; the five curated excerpts below are the explicit exception. Their selected text was masked and individually checked by the AI agent; my own sign-off is still pending. No raw names, phone numbers, emails or registrations are included in these excerpts. External AI/export privacy checks remain separate pipeline work. Tests and output-saving appear in the appendix.

## 1. What we received

**The pack contains monthly exports, quarterly exports and multi-sheet workbooks across the reporting year.** One file is not necessarily one reporting period or one dealership. Run the preparation cell below once; the parsing code lives in `investigation_support.py`.

In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import re
from collections import Counter
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p for p in candidates if (p / "feedback").is_dir() and (p / "reference").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the assignment folder or its scripts subfolder.")
FEEDBACK_DIR = PROJECT_DIR / "feedback"
OUTPUT_DIR = PROJECT_DIR / "scripts" / "investigation_outputs"

import sys
scripts_path = str(PROJECT_DIR / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from investigation_support import build_inventory, header_candidates, load_feedback_csv
from privacy_preview import safe_preview
inventory = build_inventory(PROJECT_DIR)
files = inventory['files']
file_inventory = inventory['file_inventory']
raw_grids = inventory['raw_grids']
parser_settings = inventory['parser_settings']
read_errors = inventory['read_errors']
sheet_overview = inventory['sheet_overview']
structure_records = inventory['structure_records']
audit_df = inventory['audit_df']
headers_df = inventory['headers_df']

In [ ]:
display(file_inventory.groupby("format").agg(files=("filename", "size"), bytes=("bytes", "sum")))
print(f"Total: {len(files)} files; {len(sheet_overview)} file/sheet combinations; {len(read_errors)} read failures.")
print("Full file and sheet inventories are saved by the appendix.")

## 2. Why the files don't line up

**CSV delimiters and encodings differ; worksheet headers can begin below title rows or change mid-sheet.** The loader records its settings, preserves empty fields and reports read errors. Header candidates identify where review is needed.

In [ ]:
parser_summary = pd.DataFrame([{"filename": name, **settings} for name, settings in parser_settings.items()])
display(parser_summary.groupby(["encoding", "delimiter"]).size().rename("files").reset_index())
print("Takeaway: Chatswood uses semicolons; Wollongong needs Windows-1252 decoding.")
review = audit_df.loc[audit_df["header_status"] != "candidate found", ["filename", "sheet", "candidate_rows", "header_status"]]
display(review)
print("Takeaway: multiple headers need segment mappings; sheets without a candidate need separate classification.")

**Dandenong's October header is on source row 4.** Reading row 1 as the header would treat the workbook title as a column name.

In [ ]:
example = raw_grids[("Dandenong Customer Feedback 2025-26.xlsx", "Oct 25")]
preview = safe_preview(example, header_row=3, rows=6)
preview.index = preview.index + 1
preview.index.name = "source_row"
display(preview)

## 3. Traps that change the cleaning rules

**These five masked excerpts show why scale, missingness and text interpretation matter.** The AI agent checked each displayed excerpt individually; I have not yet signed off on this selection. The comments retain their source wording apart from tagged redactions.


In [ ]:
from IPython.display import Markdown
from privacy_preview import redact_text_for_review

# Curated excerpts: source wording retained except identified spans replaced by tags.
# Reject changed source text rather than silently displaying an unreviewed replacement.
reviewed_examples = [
    {
        "file": "Geelong Feedback.xlsx",
        "sheet": "2025",
        "source_row": 5,
        "rating": "4 (1=Excellent, 5=Poor)",
        "comment_col": 3,
        "masked_comment": "Always a queue and never enough people serving. Every. Single. Time.",
        "takeaway": "A high raw number is poor-side feedback on this reversed scale."
    },
    {
        "file": "Geelong Feedback.xlsx",
        "sheet": "2025",
        "source_row": 7,
        "rating": "1 (1=Excellent, 5=Poor)",
        "comment_col": 3,
        "masked_comment": "Well, that was annoyingly painless. Where's the paperwork maze I was promised?",
        "takeaway": "The low raw number accompanies a positive experience; wording is also mildly sarcastic."
    },
    {
        "file": "Tamworth.xlsx",
        "sheet": "Sheet1",
        "source_row": 15,
        "rating": "Missing",
        "comment_col": 3,
        "masked_comment": "Ten work utes booked in over two weeks and the per-vehicle rate was lower than our usual workshop quoted. That makes the books a lot easier at tax time. ([REGO])",
        "takeaway": "The comment gives useful value-for-money evidence even though the customer supplied no rating."
    },
    {
        "file": "Tamworth.xlsx",
        "sheet": "Sheet1",
        "source_row": 56,
        "rating": "Missing",
        "comment_col": 3,
        "masked_comment": "Rang about my ute four times this week. Four. Still has the same dead battery issue they said was sorted.",
        "takeaway": "An unrated response can still support investigation of repair quality and communication."
    },
    {
        "file": "Chermside Q1 2026 FINAL v2.xlsx",
        "sheet": "Feedback",
        "source_row": 2,
        "rating": "3 (1–5 scale)",
        "comment_col": 4,
        "masked_comment": "Quick service. Call-backs? Never happened. Call me on [PHONE].",
        "takeaway": "One response can praise speed while criticising callbacks; the contact number is unnecessary for themes."
    }
]
for example in reviewed_examples:
    original = raw_grids[(example["file"], example["sheet"])].iloc[example["source_row"] - 1, example["comment_col"]]
    candidate = redact_text_for_review(original).replace("[REGISTRATION]", "[REGO]")
    if candidate != example["masked_comment"]:
        raise ValueError("A curated source excerpt changed; review its masking before display.")
    display(Markdown(
        f"**{example['file']} · {example['sheet']} · row {example['source_row']} · rating: {example['rating']}**\n\n"
        f"> {example['masked_comment']}\n\n{example['takeaway']}"
    ))


### Geelong

**Geelong reverses its rating direction between 2025 and 2026.** Preserve both the raw score and the sheet-specific interpretation.

In [ ]:
from privacy_preview import SAFE_SCALE_NOTES
scale_evidence = []
for sheet in ("2025", "2026"):
    note = str(raw_grids[("Geelong Feedback.xlsx", sheet)].iloc[1, 0])
    scale_evidence.append({"sheet": sheet, "declared_scale": note if note in SAFE_SCALE_NOTES else "Review unexpected scale note"})
display(pd.DataFrame(scale_evidence))

### Ringwood

**Ringwood changes both column positions and rating scale inside the same Q1 sheet.** January uses a five-point score in column C; from the second header at row 37, Model occupies C and the ten-point score occupies D. The note announcing the new template is on row 36.

In [ ]:
ringwood_key = ("Ringwood feedback Q1 2026.xlsx", "Feedback")
display(pd.DataFrame(header_candidates(raw_grids[ringwood_key]))[["source_row", "exact_headers"]])

### Chermside

**Chermside v2 adds six responses and preserves all 112 earlier rows unchanged.** Both versions must remain traceable while the shared responses are counted once.

In [ ]:
old_key = ("Chermside Q1 2026 FINAL.xlsx", "Feedback")
new_key = ("Chermside Q1 2026 FINAL v2.xlsx", "Feedback")
def row_multiset(grid):
    return Counter(tuple(str(v) for v in row) for row in grid.iloc[1:].itertuples(index=False, name=None))
old_rows, new_rows = row_multiset(raw_grids[old_key]), row_multiset(raw_grids[new_key])
print("Earlier version data rows:", sum(old_rows.values()))
print("v2 data rows:", sum(new_rows.values()))
print("Rows shared, including multiplicity:", sum((old_rows & new_rows).values()))
print("Rows only in earlier version:", sum((old_rows - new_rows).values()))
print("Rows only in v2:", sum((new_rows - old_rows).values()))

### Penrith

**Penrith's June tab repeats July, including its recorded dates.** Investigate the missing June coverage rather than shifting dates to fit the tab name.

In [ ]:
penrith_june = raw_grids[("Penrith Feedback.xlsx", "Jun 26")]
penrith_july = raw_grids[("Penrith Feedback.xlsx", "Jul 26")]
print("June and July grids identical:", penrith_june.equals(penrith_july))
for label, grid in (("Jun 26", penrith_june), ("Jul 26", penrith_july)):
    print(label, "first recorded date:", grid.iloc[1, 0], "physical rows:", len(grid))

### Tamworth

**Tamworth has ten responses without a rating; eight still contain a comment.** Missing customer scores stay null, while comments remain available for separately defined text analysis.

In [ ]:
tamworth = raw_grids[("Tamworth.xlsx", "Sheet1")]
tamworth_data = tamworth.iloc[1:]
missing_rating = tamworth_data.iloc[:, 2].astype(str).str.strip().eq("")
with_comment = tamworth_data.iloc[:, 3].astype(str).str.strip().ne("")
display(pd.DataFrame([{"feedback_rows": len(tamworth_data), "missing_ratings": int(missing_rating.sum()), "missing_rating_with_comment": int((missing_rating & with_comment).sum())}]))

### Toowoomba and copied Google reviews

**Toowoomba supplies aggregate counts, while Southport's copied reviews have relative dates.** These sources have a different grain or date precision from the main response table. Missing Sales rows in the summary do not establish zero Sales feedback.

In [ ]:
toowoomba = raw_grids[("Toowoomba Q4 2025 summary.xlsx", "Summary")]
reviews = raw_grids[("Southport Feedback Tracker.xlsx", "Google reviews (copied)")]
display(pd.DataFrame([
    {"source": "Toowoomba Q4 summary", "supplied_total_responses": int(toowoomba.iloc[-1, -1]), "individual_review_rows": None, "handling": "Separate aggregate table"},
    {"source": "Southport copied Google reviews", "supplied_total_responses": None, "individual_review_rows": len(reviews) - 2, "handling": "Relative dates; separate source"},
]))

## 4. Decisions

- Use a canonical response grain with dealership, date, department, rating, redacted comment and source lineage.
- Configure mappings by file, sheet and header segment. Recognise reviewed aliases and flag new values.
- Retain the original rating and scale alongside the proposed five-point representation; finalize conversion rules before comparison.
- Resolve overlapping versions and copied tabs explicitly. Keep aggregate counts separate from response-level data.
- Keep absent ratings null and report valid-rating denominators. Define AI sentiment/themes independently.
- Record provisional interpretations in `ASSUMPTIONS.md` and my/AI contributions in `AI_USAGE_LOG.md`.

**Next milestone:** implement DuckDB Bronze/Silver ingestion, reconciliation and validation. This notebook provides the findings and evidence; those transformations are not implemented here.

## Appendix: verification and saved inventories

**The checks below verify execution, preview behaviour and input accounting.** They also save the complete inventories for detailed review. The narrative above deliberately omits long signature lists and raw blank-cell tables: those counts mix headers, notes and response data and are not completeness KPIs.

### Privacy regression checks

In [ ]:
import unittest
privacy_suite = unittest.defaultTestLoader.discover(str(PROJECT_DIR / "tests"), pattern="test_privacy_preview.py")
privacy_result = unittest.TextTestRunner(verbosity=0).run(privacy_suite)
assert privacy_result.testsRun > 0 and privacy_result.wasSuccessful(), "Privacy regression checks failed."
print("Privacy regression checks passed:", privacy_result.testsRun)

### Reconciliation checks and output files

Every supported file must be accounted for, read errors must be resolved, and known parser/header differences must be retained. Save the structural inventories for the next mapping stage.

In [ ]:
accounted_files = {name for name, _ in raw_grids} | {e["filename"] for e in read_errors}
assert accounted_files == {p.name for p in files}, "Some files were not accounted for."
assert len(raw_grids) == len(sheet_overview), "Duplicate file/sheet inventory keys."
assert not read_errors, f"Resolve {len(read_errors)} read failure(s) before trusting the inventory."
assert parser_settings["chatswood_2025_q4.csv"]["delimiter"] == repr(";")
assert parser_settings["Wollongong_Oct25-Mar26.csv"]["encoding"] == "cp1252"
assert len(header_candidates(raw_grids[ringwood_key])) >= 2

OUTPUT_DIR.mkdir(exist_ok=True)
file_inventory.to_csv(OUTPUT_DIR / "file_inventory.csv", index=False)
sheet_overview.to_csv(OUTPUT_DIR / "sheet_inventory.csv", index=False)
headers_df.to_csv(OUTPUT_DIR / "header_candidates.csv", index=False)
(OUTPUT_DIR / "structure_profile.json").write_text(json.dumps(structure_records, indent=2), encoding="utf-8")
(OUTPUT_DIR / "csv_parser_settings.json").write_text(json.dumps(parser_settings, indent=2), encoding="utf-8")
print("Checks passed.")
print("Files:", len(files), "File/sheet pairs:", len(raw_grids), "Read failures:", len(read_errors))
print("Structural outputs saved to:", OUTPUT_DIR)
print("No customer-response counts or business KPIs have been claimed at this stage.")